In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import pandas as pd

geo = pd.read_csv(
    "../data/coordonnees_pluviometres_flowbru.csv"
)

print("Nombre de pluviomètres :", len(geo))

display(geo[[
    "nom",
    "latitude",
    "longitude",
    "bassin_versant"
]])

Afficher les 16 pluviomètres sur une carte simple

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Placer les stations selon leurs coordonnées GPS
ax.scatter(
    geo["longitude"],
    geo["latitude"],
    s=75,
    color="#247BA0",
    zorder=3
)

# Ajouter le nom de chaque station
for _, station in geo.iterrows():
    ax.annotate(
        station["nom"],
        (station["longitude"], station["latitude"]),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8
    )

ax.set_title(
    "Pluviomètres FlowBru — Bruxelles",
    fontsize=15,
    pad=20
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.grid(alpha=0.2)
ax.set_aspect(
    1 / np.cos(np.deg2rad(geo["latitude"].mean()))
    if False else "auto"
)

plt.tight_layout()
plt.show()

Vérifier la correspondance avec les mesures de pluie

In [ ]:
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

colonnes_pluie = mesures.columns[1:17].tolist()

# Comparer les noms de stations aux colonnes du CSV
for _, station in geo.iterrows():
    colonne = station["colonne_pluie_csv"]

    if colonne in colonnes_pluie:
        print("OK :", station["nom"])
    else:
        print("À vérifier :", station["nom"], "→", colonne)

Correction de 2 noms des pluviometre pour garder la correspondance avec les fichers CSV

In [ ]:
# Corriger les noms pour qu'ils correspondent au CSV des mesures
geo.loc[
    geo["nom"] == "Écluse Anderlecht",
    "colonne_pluie_csv"
] = "pluv.écluse;anderlecht"

# Vérifier le nom exact de la colonne IRM
print([col for col in colonnes_pluie if "irm" in col.lower()])


In [ ]:
print("IRM existe :", "pluv.IRM" in mesures.columns)

print(
    "Écluse Anderlecht existe :",
    "pluv.écluse;anderlecht" in mesures.columns
)

In [ ]:
for i, colonne in enumerate(mesures.columns):
    print(i, repr(colonne))

In [ ]:
# Recharger le fichier de mesures
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

# Corriger les deux correspondances
geo.loc[
    geo["nom"] == "IRM",
    "colonne_pluie_csv"
] = "pluv.irm"

geo.loc[
    geo["nom"] == "Écluse Anderlecht",
    "colonne_pluie_csv"
] = "pluv.écluse;anderlecht"

In [ ]:
for _, station in geo.iterrows():
    colonne = station["colonne_pluie_csv"]

    if colonne in mesures.columns:
        print("OK :", station["nom"], "=", colonne)
    else:
        print("À vérifier :", station["nom"], "=", colonne)

Pour corriger le fait que certains noms utilisent des unicode different et ne sont pas reconnus dans le CVS/dossier données

In [ ]:
import unicodedata

# Uniformiser les accents dans les deux fichiers
mesures.columns = [
    unicodedata.normalize("NFC", col)
    for col in mesures.columns
]

geo["colonne_pluie_csv"] = geo["colonne_pluie_csv"].apply(
    lambda nom: unicodedata.normalize("NFC", nom)
)

Vérifier les 16 pluviomètres

In [ ]:
for _, station in geo.iterrows():

    colonne = station["colonne_pluie_csv"]

    if colonne in mesures.columns:
        print("OK :", station["nom"], "=", colonne)
    else:
        print("À vérifier :", station["nom"], "=", colonne)

Enregistrer la correction

In [ ]:
geo.to_csv(
    "../data/coordonnees_pluviometres_flowbru.csv",
    index=False
)

print("Les correspondances sont enregistrées !")

In [ ]:
colonnes = geo["colonne_pluie_csv"]

assert len(geo) == 16, "Il manque des stations"
assert colonnes.isin(mesures.columns).all(), "Une colonne est introuvable"
assert colonnes.is_unique, "Une colonne est associée plusieurs fois"

print("✅ Les 16 pluviomètres sont correctement associés !")

Individuation d'un moment del ajournée analyse (ici 24 juillet) où il y a eu de la pluie quelque pars pour pas comencer avc tout les cercles indiquant les pluviometres trop petiti car il n'y a pas de pluie 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Trouver le moment où la pluie est la plus forte
colonnes = geo["colonne_pluie_csv"].tolist()

pluie = mesures[colonnes].apply(
    pd.to_numeric,
    errors="coerce"
)

indice = pluie.max(axis=1).idxmax()

print("Ligne sélectionnée :", indice)
print("Date et heure :", mesures.loc[indice, mesures.columns[0]])
print("Pluie maximale :", pluie.loc[indice].max())

Dessiner les cercles de la pluie

In [ ]:
# Récupérer la pluie à l'instant choisi
valeurs = pluie.loc[indice].to_numpy(dtype=float)

# Taille des cercles proportionnelle à la pluie
tailles = 30 + np.nan_to_num(valeurs, nan=0) * 100

fig, ax = plt.subplots(figsize=(10, 10))

# Dessiner les stations
ax.scatter(
    geo["longitude"],
    geo["latitude"],
    s=tailles,
    color="#247BA0",
    alpha=0.55,
    edgecolors="#16516A",
    linewidths=1
)

# Ajouter les noms
for _, station in geo.iterrows():
    ax.annotate(
        station["nom"],
        (station["longitude"], station["latitude"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8
    )

ax.set_title(
    f"Précipitations à Bruxelles\n"
    f"{mesures.loc[indice, mesures.columns[0]]}",
    fontsize=15
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.15)

# Corriger les proportions géographiques
latitude_moyenne = geo["latitude"].mean()
ax.set_aspect(1 / np.cos(np.deg2rad(latitude_moyenne)))

plt.tight_layout()
plt.show()

Vérifier les valeurs

In [ ]:
resultat = geo[["nom"]].copy()
resultat["pluie"] = valeurs

display(
    resultat.sort_values("pluie", ascending=False)
)

### Animation des 288 instants de a journée compressé en 60 secondes
24h-> 60 sec: 288 images, 4.8 images/s, 12 min réelles/s

Préparation des données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Harmoniser les accents si nécessaire
import unicodedata

mesures.columns = [
    unicodedata.normalize("NFC", col)
    for col in mesures.columns
]

geo["colonne_pluie_csv"] = geo["colonne_pluie_csv"].apply(
    lambda nom: unicodedata.normalize("NFC", nom)
)

# Colonnes de pluie dans le même ordre que les stations
colonnes = geo["colonne_pluie_csv"].tolist()

assert all(col in mesures.columns for col in colonnes)

# Matrice : 288 instants × 16 pluviomètres
pluie_animation = mesures[colonnes].apply(
    pd.to_numeric,
    errors="coerce"
).to_numpy()

print("Dimensions :", pluie_animation.shape)

Créer la carte animée

In [ ]:
# Durée de l'animation
duree = 60  # secondes
nombre_images = len(pluie_animation)
intervalle = duree * 1000 / nombre_images

# Taille des cercles
def calculer_tailles(valeurs):
    valeurs = np.nan_to_num(valeurs, nan=0)
    return 30 + valeurs * 100

# Créer la figure
fig, ax = plt.subplots(figsize=(10, 10))

# Cercles au premier instant
cercles = ax.scatter(
    geo["longitude"],
    geo["latitude"],
    s=calculer_tailles(pluie_animation[0]),
    color="#247BA0",
    alpha=0.55,
    edgecolors="#16516A",
    linewidths=1
)

# Noms des stations
for _, station in geo.iterrows():
    ax.annotate(
        station["nom"],
        (station["longitude"], station["latitude"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8
    )

# Titre mis à jour pendant l'animation
titre = ax.set_title("", fontsize=15, pad=20)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.15)

# Garder la carte fixe
ax.set_xlim(
    geo["longitude"].min() - 0.025,
    geo["longitude"].max() + 0.025
)

ax.set_ylim(
    geo["latitude"].min() - 0.015,
    geo["latitude"].max() + 0.015
)

# Corriger les proportions géographiques
latitude_moyenne = geo["latitude"].mean()
ax.set_aspect(1 / np.cos(np.deg2rad(latitude_moyenne)))

# Fonction appelée pour chaque image
def actualiser(image):
    valeurs = pluie_animation[image]

    cercles.set_sizes(calculer_tailles(valeurs))

    date_heure = mesures.iloc[image, 0]

    titre.set_text(
        f"Précipitations à Bruxelles\n{date_heure}"
    )

    return cercles, titre

# Créer l'animation
animation = FuncAnimation(
    fig,
    actualiser,
    frames=nombre_images,
    interval=intervalle,
    blit=False,
    repeat=False
)

plt.close(fig)

print("Animation créée :", nombre_images, "images")

Afficher l'animation

In [ ]:
display(
    HTML(animation.to_jshtml())
)